# NB-13-01 — An Explainability Toolkit

Build six explanation methods on the synthetic student data, then evaluate them: fidelity,
stability and a label-shuffling sanity check.

Every number printed here appears in L13-02. The methods are implemented from scratch; the
`shap` and `lime` packages are not required (compare against them as an exercise).

Run time on a 4-core laptop: about 10 seconds.

In [ ]:
# Make the module's reference code importable from the notebooks/ directory.
import sys, pathlib
CODE_DIR = None
for _base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_base / "code").is_dir() and (_base / "notebooks").is_dir():
        CODE_DIR = _base / "code"
        break
if CODE_DIR is None:
    CODE_DIR = pathlib.Path.cwd().parent / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
print("code on path:", CODE_DIR)


In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

import explain as X
from rai_data import student

d = student()
Xf, y, names = d["X"], d["y"], d["feature_names"]
X_tr, X_te, y_tr, y_te = train_test_split(Xf, y, test_size=0.3, random_state=0, stratify=y)
print("train", X_tr.shape, "test", X_te.shape, "features", names)

## 1. Two models, same features

The interpretable model is the explanation. The black box exists to show what post-hoc methods
cost.

In [ ]:
lr = LogisticRegression(max_iter=2000).fit(X_tr, y_tr)
gb = GradientBoostingClassifier(n_estimators=80, max_depth=2, random_state=0).fit(X_tr, y_tr)

for name, m in (("logistic", lr), ("boosting", gb)):
    print(f"{name:10s} accuracy {X.accuracy_at_half(m, X_te, y_te):.3f}  AUC {X.auc(m, X_te, y_te):.3f}")

print("\nlogistic coefficients (the explanation):")
for n, c in sorted(zip(names, lr.coef_[0]), key=lambda kv: -abs(kv[1])):
    print(f"  {n:20s} {c:+.3f}")

## 2. Permutation importance

AUC drop when a feature is shuffled, five repeats. Correlated features split credit, which is
why attendance looks unimportant.

In [ ]:
for row in X.permutation_importance(gb, X_te, y_te, names, n_repeats=5):
    print(f"{row['feature']:20s} {row['mean_drop']:+.4f}  (sd {row['std']:.4f})")

## 3. Partial dependence

Average prediction as attendance sweeps its observed range.

In [ ]:
j = names.index("attendance_rate")
grid, values = X.partial_dependence(gb, X_te, j, n_grid=6)
for g, v in zip(grid, values):
    print(f"attendance {g:.2f} -> mean predicted pass {v:.3f}")

## 4. Exact Shapley values

Enumerate all $2^4$ coalitions. The efficiency check `sum(phi) == f(x) - E[f]` is the test that
catches a broken implementation.

In [ ]:
probs = X.probability(gb, X_te)
idx = int(np.argsort(np.abs(probs - 0.5))[0])
x = X_te[idx]
print(f"instance {idx}, predicted {probs[idx]:.3f}")

sh = X.shapley_values(gb, x, X_tr, names)
print(f"base {sh['base_value']:.4f}  sum(phi) {sh['sum_phi']:+.4f}  "
      f"expected {sh['expected_sum']:+.4f}  gap {sh['efficiency_gap']:.2e}")
for f, v in zip(sh["features"], sh["values"]):
    print(f"  {f:20s} {v:+.4f}")

## 5. LIME and its fidelity

Fidelity is the number most reports omit. A local $R^2$ of 0.26 means the linear story explains a
quarter of the black box's local behaviour.

In [ ]:
lim = X.lime_tabular(gb, x, X_tr, names)
for f, v in zip(lim["features"], lim["values"]):
    print(f"  {f:20s} {v:+.3f}")
print(f"local R^2 {lim['local_r2']:.3f}")

stab = X.explanation_stability(gb, x, X_tr, names)
print(f"stability over 5 seeds: mean r {stab['mean_correlation']:.3f} "
      f"(min {stab['min']:.3f}, max {stab['max']:.3f})")
print("\nShapley vs LIME ranking:")
print("  shapley:", sh["features"])
print("  lime   :", lim["features"])

## 6. Counterfactual

The nearest input change that flips the decision. Report the distance: a decision that turns on
0.03 standard deviations is fragile.

In [ ]:
p = float(X.probability(gb, x[None, :])[0])
cf = X.counterfactual(gb, x, X_tr, names, desired=1 if p < 0.5 else 0)
print("found:", cf["found"])
if cf["found"]:
    for ch in cf["changes"]:
        print(f"  {ch['feature']:20s} {ch['from']:.2f} -> {ch['to']:.2f}")
    print(f"  distance {cf['distance']:.3f} sd, new probability {cf['new_probability']:.3f}")

## 7. Sanity check

Retrain on shuffled labels. A correct method must collapse to near-zero importance.

In [ ]:
chk = X.sanity_check(Xf, y, names)
print(f"real      AUC {chk['real_auc']:.3f}  max importance {chk['real_max']:+.4f}")
print(f"shuffled  AUC {chk['shuffled_auc']:.3f}  max importance {chk['shuffled_max']:+.4f}")
assert chk["real_max"] > 5 * chk["shuffled_max"], "importance method may be measuring the data"
print("\npass: real importance exceeds shuffled by more than 5x")

## Exercises

1. Replace exact enumeration with a sampled KernelSHAP and plot the efficiency gap against the
   number of samples. How many samples do you need before the gap falls below 0.01?
2. Raise LIME's `n_samples` to 4000. Report the change in local $R^2$ and stability.
3. Find the instance furthest from the decision boundary and repeat sections 4–6. Which
   explanation changes most?
4. Add a fifth feature that is a copy of `attendance_rate` plus noise. What happens to both
   features' Shapley values, and to permutation importance?
5. Run the sanity check with a model trained on 5% of the data. Does the shuffled-label
   importance stay near zero?